In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the cleaned dataset
project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
cleaned_file = data_folder / "propertylens_cleaned.csv"

df = pd.read_csv(cleaned_file)

print("Rows:", len(df))
print("Columns:", len(df.columns))


# Price in easier units
df["price_lakh"] = df["price"] / 100000
df["price_crore"] = df["price"] / 10000000


# Keep area in square feet with a clearer name
df["area_sqft"] = df["area"]


# Group BHK values
df["bhk_group"] = df["bhk_count"].astype("string")

df.loc[df["bhk_count"] == 0, "bhk_group"] = "Not specified"
df.loc[df["bhk_count"] >= 8, "bhk_group"] = "8+ BHK"


# Price bands based on price in lakh
df["price_band"] = pd.cut(
    df["price_lakh"],
    bins=[0, 50, 100, 200, 500, 1000, np.inf],
    labels=[
        "Below 50L",
        "50L-1Cr",
        "1Cr-2Cr",
        "2Cr-5Cr",
        "5Cr-10Cr",
        "Above 10Cr"
    ],
    include_lowest=True
)


# Area bands
df["area_band"] = pd.cut(
    df["area_sqft"],
    bins=[0, 1000, 1500, 2000, 3000, 5000, np.inf],
    labels=[
        "Below 1000 sqft",
        "1000-1500 sqft",
        "1500-2000 sqft",
        "2000-3000 sqft",
        "3000-5000 sqft",
        "Above 5000 sqft"
    ],
    include_lowest=True
)


# Check the new columns
new_columns = [
    "price_lakh",
    "price_crore",
    "area_sqft",
    "bhk_group",
    "price_band",
    "area_band"
]

print("\nNew columns:")
print(new_columns)


# Check the new values
print("\nPrice in lakh:")
print(df["price_lakh"].describe())

print("\nPrice in crore:")
print(df["price_crore"].describe())

print("\nBHK groups:")
print(df["bhk_group"].value_counts(dropna=False).sort_index())

print("\nPrice bands:")
print(df["price_band"].value_counts(sort=False))

print("\nArea bands:")
print(df["area_band"].value_counts(sort=False))


# Check for missing values created by feature engineering
print("\nMissing values in new columns:")
print(df[new_columns].isna().sum())


# Save the feature-engineered dataset
feature_file = data_folder / "propertylens_featured.csv"

df.to_csv(feature_file, index=False)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 14225
Columns: 12

New columns:
['price_lakh', 'price_crore', 'area_sqft', 'bhk_group', 'price_band', 'area_band']

Price in lakh:
count    14225.000000
mean       399.505392
std        575.495358
min          0.950000
25%        141.000000
50%        262.000000
75%        455.000000
max      12263.000000
Name: price_lakh, dtype: float64

Price in crore:
count    14225.000000
mean         3.995054
std          5.754954
min          0.009500
25%          1.410000
50%          2.620000
75%          4.550000
max        122.630000
Name: price_crore, dtype: float64

BHK groups:
bhk_group
1                 251
2                2337
3                6151
4                3724
5                 587
6                  89
7                  16
8+ BHK             71
Not specified     999
Name: count, dtype: int64[pyarrow]

Price bands:
price_band
Below 50L      908
50L-1Cr       1241
1Cr-2Cr       3343
2Cr-5Cr       5578
5Cr-10Cr      2315
Above 10Cr     840
Name: count, dtype: int64

Area 